# 📈 Avaliação Final no Test Set (15%) e Explicabilidade

In [ ]:
import sys, os, warnings
sys.path.append('../src')
warnings.filterwarnings('ignore')
import pandas as pd
import joblib, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, RocCurveDisplay
from features import prepare_datasets, load_feature_list
from models import split_dataset, final_evaluation, feature_importance

df = pd.read_csv('../data/processed/ransom_cleaned.csv')
X, y, le, groups = prepare_datasets(df, return_groups=True)
X_train, X_val, X_test, y_train, y_val, y_test, _ = split_dataset(X, y, groups, verbose=False)
model = joblib.load('../models/ransomware_model.pkl')
assert list(X.columns) == load_feature_list('../models/ransomware_features.json')

In [ ]:
print(classification_report(y_test, model.predict(X_test), target_names=le.classes_, digits=4))
print({k: round(v, 4) if isinstance(v, float) else v for k, v in final_evaluation(model, X_test, y_test).items()})

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, display_labels=le.classes_, ax=ax[0], colorbar=False)
RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax[1])
plt.tight_layout(); plt.show()

## Importância das features (Top 15)
⚠️ `processes_malicious` domina (~79% da importância): aparece em 99,8% do malware e em 1,9% do benigno. É uma feature **dinâmica** que exige execução em sandbox. Sem ela o F1 desce de 0,995 para 0,988, e só com features estáticas para 0,951.

In [ ]:
top = feature_importance(model, X.columns, 15)
plt.figure(figsize=(10, 6)); sns.barplot(x=top.values, y=top.index, color='teal')
plt.title('Top 15 features'); plt.tight_layout(); plt.show()